# alternance-extractor -- QLoRA fine-tune (Qwen2.5-1.5B-Instruct)

**Status: draft skeleton, not yet run.** Written before `data/train/train.jsonl` exists --
that file is produced by `label/select_test_set.py`, which itself needs the full 642-posting
Groq labelling run (`label/run_labelling.py`) to finish first. See the repo README for the
full pipeline order.

**Claim being tested**: a QLoRA fine-tune of Qwen2.5-1.5B-Instruct on Groq-labelled postings
can match Groq's Llama-3.3-70B-class baseline on structured extraction, at lower cost/latency.
This notebook does the fine-tuning; `notebooks/kaggle_benchmark.ipynb` (not built yet) will
score both models with `eval/score.py` against the hand-corrected `data/test/test.jsonl`.

**Inputs this notebook expects** (upload as a Kaggle Dataset, or clone the repo if it's
public and `data/train/train.jsonl` has been committed):
- `schema/posting.py`, `label/prompt.py` -- reused so the fine-tuning prompt can never drift
  from the extraction schema or the Groq-labelling prompt.
- `data/train/train.jsonl` -- Groq-labelled training postings.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

## Repo access

Clones the repo so `schema.posting` and `label.prompt` can be imported directly, instead of
hand-duplicating the schema/prompt here and risking drift. If you've instead attached the repo
as a Kaggle Dataset input, skip the clone and just set `REPO_ROOT` to that input path.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mmattar18/alternance-extractor.git"
REPO_ROOT = Path("/kaggle/working/alternance-extractor")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

from label.prompt import SYSTEM_PROMPT  # noqa: E402  -- same schema/rules text used for Groq labelling

## Config

`MAX_SEQ_LEN` should comfortably cover `SYSTEM_PROMPT` + the longest `raw_text` + the JSON
target. Postings are pre-filtered to `description_chars >= 200` at ingest time but have no
upper bound, so check the actual token-length distribution once `train.jsonl` exists and raise
this if there's meaningful truncation -- don't just assume 2048 is enough.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_PATH = REPO_ROOT / "data" / "train" / "train.jsonl"
OUTPUT_DIR = "/kaggle/working/qlora-adapter"
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
SEED = 42

## Load training data

Each row in `train.jsonl` is a labelled posting: original ingest fields plus Groq's `prediction`
(already schema-validated JSON, or `None` for the rows `select_test_set.py` dropped as invalid).
Training target is `json.dumps(prediction)` -- the exact string a correct extraction should
produce, so inference-time parsing (`schema.posting.parse_llm_json`) stays comparable to how
Groq's output was parsed during labelling.

Deliberately **not** including the few-shot examples from `label/prompt.py` here: those exist
to give Groq in-context examples it was never trained on. A fine-tuned model sees hundreds of
real examples via gradient updates instead, so baking the same 3 examples into every training
row would only burn context length for no benefit. `SYSTEM_PROMPT` (schema + extraction rules)
is reused as-is so the instructions stay identical to what labelled the data in the first place.

In [ ]:
records = [json.loads(l) for l in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"{len(records)} training postings loaded")

def to_chat_example(r):
    target = json.dumps(r["prediction"], ensure_ascii=False)
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r["raw_text"]},
            {"role": "assistant", "content": target},
        ]
    }

dataset = Dataset.from_list([to_chat_example(r) for r in records])
dataset = dataset.train_test_split(test_size=0.05, seed=SEED)  # small held-out slice for eval_loss during training, distinct from data/test/test.jsonl
dataset

## Tokenizer, quantized base model, LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

`packing=False`: each example is one posting -> one extraction, and packing multiple postings
into one sequence would let the model attend across unrelated postings, which we don't want for
this task.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer.train()

## Save adapter

Saves the LoRA adapter only (small). Merge into the base model at inference time in
`notebooks/kaggle_benchmark.ipynb`, or with `model.merge_and_unload()` here if a single merged
checkpoint is more convenient for wherever benchmarking runs.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

## Next step

Run `notebooks/kaggle_benchmark.ipynb` (not built yet) to generate predictions from this
adapter and the Groq baseline over `data/test/test.jsonl`, then score both with
`eval/score.py` -- that produces the three comparison numbers the README's claim needs:
field-level F1, exact-match rate, and JSON-validity rate, at whatever cost/latency each model
actually measured at.